# Irodori-TTS-600M-v3-VoiceDesign on Google Colab

日本語テキストから、キャプション（テキスト説明）や参照音声で声質を指定して音声を生成するTTSモデル（v3）。

**v3の特徴**: 参照音声で声をクローンしつつ、キャプションで話し方を同時に制御できます（3要素制御）。

**必要**: GPUランタイム（T4以上推奨）

> メニュー → ランタイム → ランタイムのタイプを変更 → GPU を選択

In [ ]:
# GPU確認
!nvidia-smi

In [ ]:
# Irodori-TTS リポジトリとWebUIコードをクローン
!git clone https://github.com/Aratako/Irodori-TTS.git
!git clone https://github.com/shinshin86/Irodori-TTS-600M-v3-VoiceDesign-with-colab.git
!cp Irodori-TTS-600M-v3-VoiceDesign-with-colab/simple_app.py Irodori-TTS/
%cd Irodori-TTS

In [ ]:
# 依存関係のインストール
!pip install -r requirements.txt -q
!pip install protobuf>=5.26.1 -q

## WebUI (Gradio) の起動

cloudflaredトンネルで公開URLを発行します。表示されたURLをクリックしてWebUIにアクセスしてください。

**機能:**
- テキスト入力で日本語音声を合成
- キャプションで声質・感情・話し方を指定
- 参照音声をアップロードして声をクローン（キャプションと同時利用可）

In [ ]:
# cloudflared をインストールしてトンネルで公開URLを発行
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import subprocess, time, threading, re

# WebUIをバックグラウンドで起動
server = subprocess.Popen(["python", "simple_app.py"])
time.sleep(5)

# cloudflared トンネルで公開
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:7861"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

def print_url():
    for line in tunnel.stderr:
        m = re.search(r"(https://[a-z0-9-]+\.trycloudflare\.com)", line)
        if m:
            print(f"\n{'='*60}")
            print(f"WebUI URL: {m.group(1)}")
            print(f"{'='*60}\n")
            break

t = threading.Thread(target=print_url, daemon=True)
t.start()
t.join(timeout=30)

try:
    server.wait()
except KeyboardInterrupt:
    server.terminate()
    tunnel.terminate()